# Lesson 19 Lab — Numerical Stability and Cast Boundaries

**Puzzle:** When max shifting, FP32 accumulation, overflow, and underflow change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates max shifting, FP32 accumulation, overflow, and underflow and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Low-precision storage does not require low-precision reduction. Stable Softmax subtracts the row maximum before exp and normally accumulates the denominator in FP32. Each cast boundary is therefore an algorithm choice, not formatting.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["max shifting, FP32 accumulation, overflow, and underflow"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Small random inputs may never exercise overflow, underflow, cancellation, or long-reduction error.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 19
LESSON_TITLE = 'Numerical Stability and Cast Boundaries'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260832
}


## 5. Freeze the experiment

**Experiment:** Feed large positive FP16 logits to unshifted eager Softmax and max-shifted Triton Softmax.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 2096519,
  "secondary": 0.00024390220642089844,
  "max_abs_error": 0.00024390220642089844,
  "passed": true,
  "details": {
    "stable_invalid": 0,
    "input_min": -14.984375,
    "input_max": 174.25
  }
}
The unshifted FP16 expression produced 2,096,519 non-finite values. Max-shifted Triton softmax produced none and differed from FP32 reference by 2.439e-04.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Unstable non-finite values | 2,096,519 |
| Stable max error | 2.439e-04 |
| Maximum absolute error | 2.439e-04 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The unshifted FP16 expression produced 2,096,519 non-finite values. Max-shifted Triton softmax produced none and differed from FP32 reference by 2.439e-04.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Gate optimized kernels with adversarial ranges, NaN/Inf counts, extrema, and a higher-precision reference.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 19,
  "title": "Numerical Stability and Cast Boundaries",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260832
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 2096519,
    "secondary": 0.00024390220642089844,
    "max_abs_error": 0.00024390220642089844,
    "passed": true,
    "details": {
      "stable_invalid": 0,
      "input_min": -14.984375,
      "input_max": 174.25
    }
  },
  "analysis_en": "The unshifted FP16 expression produced 2,096,519 non-finite values. Max-shifted Triton softmax produced none and differed from FP32 reference by 2.439e-04.",
  "analysis_zh": "未减最大值的 FP16 表达式产生 2,096,519 个非有限值；max-shift Triton softmax 没有非有限值，相对 FP32 参考最大误差 2.439e-04。",
  "conclusion": "Gate optimized kernels with adver

## 10. Make the bounded decision

> Gate optimized kernels with adversarial ranges, NaN/Inf counts, extrema, and a higher-precision reference.

**Failure analysis:** Small random inputs may never exercise overflow, underflow, cancellation, or long-reduction error.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
